# TransHLA — Phase 3: Run Experiment

Huấn luyện TransHLA và các mô hình đối chứng trên embedding đã tính ở Phase 2,
rồi tái hiện **Bảng 2** (benchmark), **Hình 3** (ROC/PR) và phần **case study** trên
CEDAR / VDJdb / ImmuneCode / dbPepNeo2.0.

**Cần GPU.**

### Kiến trúc (mục *The architectures of the deep learning model*)

```
peptide ──ESM2──┬── sequence embedding E (1280×L) ──┬── Transformer encoder ──┐
                │                                   └── Residual CNN #1 ──────┤── concat ── MLP ── 2 lớp
                └── contact map S (L×L) ───────────────── Residual CNN #2 ─────┘
```

* **CNN Module 1**: region embedding → conv block (2 lớp conv, 256 feature map)
* **CNN Module 2**: pooling giảm một nửa độ dài → conv block, lặp tới khi độ dài = 1, kết thúc bằng BatchNorm
* Kết nối tắt (residual) giữa các module để chống tiêu biến gradient
* **TIM loss**: `L = CE − H(Y) + α·H(Y|X)` với `α = 0.04`

## 0. Đồng bộ code từ GitHub

Code được sửa trong **VS Code** trên máy, `git push` lên GitHub, ô dưới đây kéo bản mới nhất về Colab.
Repo công khai nên chạy được ngay, không cần token.

> Ô này **ghi đè** mọi sửa đổi phía Colab. Nguồn sự thật là GitHub.


In [ ]:
# === Đồng bộ code từ GitHub — chạy ô này đầu tiên, mỗi phiên Colab ===
REPO = "github.com/Tienanh209/TransHLA_repro.git"
BRANCH = "main"

import os, sys, pathlib, subprocess

def _sh(*a):
    return subprocess.run(a, capture_output=True, text=True)

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN") or ""
except Exception:
    pass

url = f"https://{token + '@' if token else ''}{REPO}"
WORK = pathlib.Path("/content/TransHLA_repro")

if (WORK / ".git").exists():
    _sh("git", "-C", str(WORK), "remote", "set-url", "origin", url)
    _sh("git", "-C", str(WORK), "fetch", "origin", BRANCH)
    _sh("git", "-C", str(WORK), "reset", "--hard", f"origin/{BRANCH}")
else:
    r = _sh("git", "clone", "--branch", BRANCH, url, str(WORK))
    if r.returncode:
        raise SystemExit((r.stderr.replace(token, "***") if token else r.stderr))

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code tại:", WORK, "|", _sh("git", "-C", str(WORK), "log", "-1", "--oneline").stdout.strip())


## 0. Cấu hình

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
!pip -q install pyarrow scikit-learn tqdm

In [ ]:
import pathlib, json

PROJ = pathlib.Path('/content/drive/MyDrive/s1156052/EpitopePrediction')
DATA, CKPT, RES = PROJ/'data', PROJ/'checkpoints', PROJ/'results'
for p in (CKPT, RES): p.mkdir(parents=True, exist_ok=True)

# ---- tham số ---------------------------------------------------------------
HLA_CLASS      = 'I'        # 'I' hoặc 'II'
CACHE_ON_DRIVE = True       # phải khớp với Phase 2
EPOCHS         = 20
BATCH          = 256
LR             = 1e-4
ALPHA_TIM      = 0.04       # hệ số alpha của TIM loss (theo bài báo)
PATIENCE       = 5          # early stopping theo AUC trên tập validation
TRAIN_BASELINES = True      # huấn luyện cả TextCNN/TextRCNN/DPCNN/RNN-ATTs (Bảng 2)
# ---------------------------------------------------------------------------

CACHE = ((PROJ/'cache') if CACHE_ON_DRIVE else pathlib.Path('/content/cache')) / f'hla{HLA_CLASS}'
meta = json.loads((CACHE/'meta.json').read_text())
N, MAXLEN, EMB_DIM = meta['n'], meta['maxlen'], meta['emb_dim']
print(meta)

## 1. Định nghĩa mô hình

### Kiến trúc mô hình

Code mô hình nằm trong **`models_transhla.py`** của repo — sửa file đó trong VS Code, `git push`,
rồi chạy lại ô "Đồng bộ code từ GitHub" ở trên để Colab nhận bản mới.

*(Trước đây ô này dùng `%%writefile` để sinh ra file; nay file được quản lý bằng git nên ô đó đã bỏ.)*


In [ ]:
import importlib, models_transhla
importlib.reload(models_transhla)
from models_transhla import TransHLA, TIMLoss, MODELS
import torch

m = TransHLA(seq_len=MAXLEN)
print(f'TransHLA: {sum(p.numel() for p in m.parameters())/1e6:.2f}M tham số (chưa tính ESM2)')
e = torch.randn(4, MAXLEN, EMB_DIM); c = torch.randn(4, MAXLEN, MAXLEN)
print('đầu ra thử:', m(e, c).shape)
del m

## 2. Dataset đọc từ memmap

In [ ]:
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader

idx_df = pd.read_parquet(CACHE/'index.parquet')
assert len(idx_df) == N

emb_mm = np.memmap(CACHE/'emb.f16',     dtype=np.float16, mode='r', shape=(N, MAXLEN, EMB_DIM))
con_mm = np.memmap(CACHE/'contact.f16', dtype=np.float16, mode='r', shape=(N, MAXLEN, MAXLEN))

class EmbDataset(Dataset):
    def __init__(self, rows):
        self.rows = np.asarray(rows)
        self.y = idx_df['label'].values[self.rows].astype(np.int64)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return (torch.from_numpy(np.asarray(emb_mm[r], dtype=np.float32)),
                torch.from_numpy(np.asarray(con_mm[r], dtype=np.float32)),
                torch.tensor(self.y[i]))

rows = {s: np.where(idx_df['split'].values == s)[0] for s in ('train','val','test','external')}
for k, v in rows.items():
    print(f'{k:9s}: {len(v):>8,}')

def loader(split, shuffle, bs=BATCH):
    return DataLoader(EmbDataset(rows[split]), batch_size=bs, shuffle=shuffle,
                      num_workers=2, pin_memory=True, drop_last=shuffle)

train_dl, val_dl, test_dl = loader('train', True), loader('val', False), loader('test', False)

## 3. Hàm đánh giá (các công thức 12–17)

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, confusion_matrix)

def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {
        'ACC':         accuracy_score(y_true, y_pred) * 100,
        'F1':          f1_score(y_true, y_pred) * 100,
        'Recall':      recall_score(y_true, y_pred) * 100,
        'Precision':   precision_score(y_true, y_pred, zero_division=0) * 100,
        'Specificity': tn / (tn + fp) * 100 if (tn+fp) else 0.0,
        'MCC':         matthews_corrcoef(y_true, y_pred),
        'AUC':         roc_auc_score(y_true, y_prob) * 100,
        'AUPRC':       average_precision_score(y_true, y_prob) * 100,
    }

@torch.no_grad()
def predict(model, dl, device='cuda'):
    model.eval(); probs, ys = [], []
    for emb, con, y in dl:
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(emb.to(device, non_blocking=True), con.to(device, non_blocking=True))
        probs.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(probs)

## 4. Vòng huấn luyện

In [ ]:
import time
from tqdm.auto import tqdm

device = 'cuda'

def train_model(name, epochs=EPOCHS, lr=LR, use_tim=True, tag=''):
    torch.manual_seed(42)
    model = MODELS[name](seq_len=MAXLEN, emb_dim=EMB_DIM).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda')
    crit = TIMLoss(ALPHA_TIM) if use_tim else None
    ce = torch.nn.CrossEntropyLoss()

    best_auc, bad, hist = -1, 0, []
    ckpt_path = CKPT / f'{name}{tag}_hla{HLA_CLASS}.pt'

    for ep in range(1, epochs+1):
        model.train(); t0 = time.time(); tot = 0.0; nb = 0
        for emb, con, y in tqdm(train_dl, desc=f'{name} ep{ep}', leave=False):
            emb, con, y = emb.to(device, non_blocking=True), con.to(device, non_blocking=True), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(emb, con)
                loss = crit(logits, y)[0] if crit else ce(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            tot += loss.item(); nb += 1
        sched.step()

        yv, pv = predict(model, val_dl)
        mv = compute_metrics(yv, pv)
        hist.append({'epoch': ep, 'loss': tot/nb, **mv, 'sec': time.time()-t0})
        print(f'  ep{ep:>2}  loss {tot/nb:.4f}  val ACC {mv["ACC"]:.2f}  AUC {mv["AUC"]:.2f}  '
              f'({time.time()-t0:.0f}s)')

        if mv['AUC'] > best_auc:
            best_auc, bad = mv['AUC'], 0
            torch.save({'state_dict': model.state_dict(), 'epoch': ep, 'val': mv,
                        'maxlen': MAXLEN, 'emb_dim': EMB_DIM}, ckpt_path)
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'  early stop ở epoch {ep}'); break

    model.load_state_dict(torch.load(ckpt_path)['state_dict'])
    pd.DataFrame(hist).to_csv(RES / f'history_{name}{tag}_hla{HLA_CLASS}.csv', index=False)
    return model, pd.DataFrame(hist)

In [ ]:
transhla, hist = train_model('TransHLA')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(hist.epoch, hist.loss, marker='o'); ax[0].set_title('TIM loss (train)'); ax[0].set_xlabel('epoch')
ax[1].plot(hist.epoch, hist.ACC, marker='o', label='ACC')
ax[1].plot(hist.epoch, hist.AUC, marker='s', label='AUC')
ax[1].set_title('validation'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Bảng 2 — so sánh với các mô hình phân loại chuỗi

In [ ]:
results, curves = {}, {}

yt, pt = predict(transhla, test_dl)
results['TransHLA'] = compute_metrics(yt, pt); curves['TransHLA'] = (yt, pt)
print('TransHLA:', {k: round(v, 3) for k, v in results['TransHLA'].items()})

if TRAIN_BASELINES:
    for name in ['TextCNN', 'TextRCNN', 'DPCNN', 'RNN-ATTs']:
        print(f'\n=== {name} ===')
        mdl, _ = train_model(name, use_tim=False)   # đối chứng dùng cross-entropy chuẩn
        y, p = predict(mdl, test_dl)
        results[name] = compute_metrics(y, p); curves[name] = (y, p)
        print({k: round(v, 3) for k, v in results[name].items()})
        del mdl; torch.cuda.empty_cache()

In [ ]:
tab = pd.DataFrame(results).T[['ACC','F1','Recall','MCC','AUC']].round(2)
tab.index.name = f'HLA-{HLA_CLASS}'
display(tab)

PAPER = {
 'I':  {'TransHLA':[84.72,84.59,83.92,0.69,91.95], 'TextCNN':[81.63,79.61,83.50,0.63,89.37],
        'TextRCNN':[81.21,76.65,83.78,0.64,87.62], 'DPCNN':[83.75,83.76,83.96,0.68,90.97],
        'RNN-ATTs':[81.17,82.27,81.46,0.63,87.98]},
 'II': {'TransHLA':[79.94,81.07,86.19,0.60,88.14], 'TextCNN':[73.26,73.49,72.64,0.47,80.64],
        'TextRCNN':[70.96,72.02,69.28,0.42,78.21], 'DPCNN':[77.41,77.91,75.98,0.55,85.30],
        'RNN-ATTs':[69.04,69.83,67.89,0.38,75.81]},
}
paper = pd.DataFrame(PAPER[HLA_CLASS], index=['ACC','F1','Recall','MCC','AUC']).T
cmp = tab.join(paper, rsuffix='_paper', how='left')
print('\n=== Tái hiện  vs  Bài báo ===')
display(cmp)
cmp.to_csv(RES / f'table2_hla{HLA_CLASS}.csv')

## 6. Hình 3 — đường cong ROC và PR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for name, (y, p) in curves.items():
    fpr, tpr, _ = roc_curve(y, p)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y,p):.3f})')
    pr, rc, _ = precision_recall_curve(y, p)
    axes[1].plot(rc, pr, label=f'{name} (AP={average_precision_score(y,p):.3f})')
axes[0].plot([0,1],[0,1],'k--',lw=.8)
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].set_title(f'ROC — HLA-{HLA_CLASS} (IEDB test)'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision–Recall — HLA-{HLA_CLASS}'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(RES / f'fig3_hla{HLA_CLASS}.png', dpi=200); plt.show()

## 7. Nghiên cứu ablation (Hình S1)

Bỏ lần lượt từng module để đo đóng góp của nó vào AUROC/AUPRC.

In [ ]:
import torch.nn as nn
from models_transhla import TransHLA as _T

class Ablated(_T):
    def __init__(self, *a, drop=None, **kw):
        super().__init__(*a, **kw); self.drop_branch = drop
    def forward(self, emb, contact, return_repr=False):
        t  = self.transformer(emb)
        c1 = self.cnn_seq(emb.transpose(1, 2))
        c2 = self.cnn_con(contact)
        if self.drop_branch == 'transformer': t  = torch.zeros_like(t)
        if self.drop_branch == 'cnn_seq':     c1 = torch.zeros_like(c1)
        if self.drop_branch == 'contact':     c2 = torch.zeros_like(c2)
        rep = torch.cat([t, c1, c2], dim=-1)
        return self.head(rep)

RUN_ABLATION = False     # bật lên nếu muốn chạy (mỗi biến thể = 1 lần huấn luyện đầy đủ)
if RUN_ABLATION:
    abl = {}
    for drop in [None, 'transformer', 'cnn_seq', 'contact']:
        torch.manual_seed(42)
        MODELS['_abl'] = lambda seq_len, emb_dim, d=drop: Ablated(seq_len=seq_len, emb_dim=emb_dim, drop=d)
        mdl, _ = train_model('_abl', epochs=max(5, EPOCHS//2), tag=f'_no_{drop}')
        y, p = predict(mdl, test_dl)
        abl[drop or 'full'] = compute_metrics(y, p)
        del mdl; torch.cuda.empty_cache()
    display(pd.DataFrame(abl).T[['ACC','AUC','AUPRC']].round(2))

## 8. Case study trên tập external

Bài báo đánh giá TransHLA trên CEDAR, VDJdb, ImmuneCode, dbPepNeo2.0 (Bảng 3–4) và cho biết
đạt **90,65% độ chính xác** trên tập neoepitope đã được kiểm chứng bằng thực nghiệm TCR (NEPDB).
Điểm mạnh chính so với các công cụ dự đoán ái lực là **specificity** cao hơn hẳn — tức ít
gán nhầm peptide thường thành epitope.

In [ ]:
if len(rows['external']):
    ext_dl = loader('external', False)
    ye, pe = predict(transhla, ext_dl)
    ext_df = idx_df.iloc[rows['external']].copy()
    ext_df['prob'] = pe
    ext_df['pred'] = (pe >= 0.5).astype(int)

    out = []
    for src, g in ext_df.groupby(ext_df['source'].str.replace('_random','',regex=False)):
        if g['label'].nunique() < 2:
            continue
        out.append({'dataset': src, 'n': len(g),
                    **compute_metrics(g['label'].values, g['prob'].values)})
    ext_tab = pd.DataFrame(out).round(2)
    display(ext_tab)
    ext_tab.to_csv(RES / f'case_study_hla{HLA_CLASS}.csv', index=False)
    ext_df.to_csv(RES / f'external_predictions_hla{HLA_CLASS}.csv', index=False)
else:
    print('Không có dữ liệu external — bỏ qua (xem lại mục 3 của Phase 1).')

## 9. Đối chiếu với trọng số chính thức trên HuggingFace

Bước kiểm tra chéo quan trọng: chạy **model gốc của tác giả** trên đúng tập test của bạn.
Nếu model gốc đạt ≈ số liệu trong bài báo mà bản tái hiện thấp hơn nhiều, vấn đề nằm ở
phần huấn luyện/siêu tham số, chứ không phải ở dữ liệu.

In [ ]:
!pip -q install fair-esm transformers

In [ ]:
from transformers import AutoTokenizer, AutoModel

REPO = f'SkywalkerLu/TransHLA_{HLA_CLASS}'
tok = AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
official = AutoModel.from_pretrained(REPO, trust_remote_code=True).to(device).eval()

def pad_to_length(seqs, target_length=MAXLEN):
    for s in seqs:
        if len(s) < target_length:
            s.extend([1] * (target_length - len(s)))
    return seqs

@torch.no_grad()
def official_predict(peptides, bs=64):
    probs = []
    for i in tqdm(range(0, len(peptides), bs), desc='official'):
        enc = pad_to_length(tok(peptides[i:i+bs])['input_ids'])
        outputs, _ = official(torch.tensor(enc).to(device))
        probs.append(torch.softmax(outputs.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(probs)

sub = idx_df.iloc[rows['test']].sample(min(5000, len(rows['test'])), random_state=0)
p_off = official_predict(sub['peptide'].tolist())
m_off = compute_metrics(sub['label'].values, p_off)
print('Trọng số chính thức trên tập test của bạn:',
      {k: round(v, 2) for k, v in m_off.items()})
print('Bản tái hiện (toàn bộ test):',
      {k: round(v, 2) for k, v in results['TransHLA'].items()})

## 10. Lưu kết quả

In [ ]:
summary = {
    'hla_class': HLA_CLASS, 'n_samples': int(N), 'maxlen': MAXLEN,
    'epochs': EPOCHS, 'batch': BATCH, 'lr': LR, 'alpha_tim': ALPHA_TIM,
    'results': {k: {kk: float(vv) for kk, vv in v.items()} for k, v in results.items()},
}
(RES / f'summary_hla{HLA_CLASS}.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary['results'], indent=2))
print('\nKết quả đã lưu ở', RES)

---

### Nếu kết quả thấp hơn bài báo, kiểm tra theo thứ tự

1. **Kích thước dữ liệu** — `SUBSET_N = 150_000` cho ACC thấp hơn bản đầy đủ khoảng 1–3 điểm. Muốn khớp số liệu thì phải dùng toàn bộ dữ liệu.
2. **Negative samples** — negative sinh ngẫu nhiên *dễ* hơn nhiều so với negative cắt từ protein nguồn (DIAMOND + CD-HIT). Dùng bộ negative của tác giả nếu có.
3. **Số epoch** — bài báo không công bố số epoch; thử 30–50 epoch với early stopping.
4. **α của TIM loss** — bài báo đặt `α = 0.04`; thử quét 0.01 / 0.04 / 0.1.
5. **Lớp biểu diễn của ESM2** — mặc định dùng layer 33 (layer cuối); thử layer 32.
6. **Số lớp Transformer** — bài báo không nêu; thử `n_tf_layers = 2`.